# Experimento pareado con tres semillas: LSTM32 solo telemetría frente a telemetría + estrategia — Colab CPU

El diagnóstico con gradient boosting (`docs/aclaraciones/29_diagnostico_gradient_boosting.md`) mostró que el estado de la estrategia explica la entrada a boxes mucho mejor que la telemetría del coche (AP 0,269 → 0,333 en LightGBM). Este notebook comprueba si la LSTM también aprovecha esa información.

| Hijo | Entradas |
|---|---|
| `lstm32_solo_telemetria` (referencia) | `muestras_preprocesadas_v1_3_telemetria_v1`: 15 numéricas, `Compound_asof`, `track_phase_asof`, sin contexto estático. |
| `lstm32_telemetria_estrategia` (candidata) | Lo mismo + 9 variables de estrategia por vuelta de la ventana: `tyre_life_vs_typical_stint`, `laps_since_last_stop`, `pace_vs_stint_best`, `field_pit_entries_last2laps`, `laps_remaining`, `gap_ahead_s`, `race_fraction`, `stops_done`, `two_dry_compounds_used`. |

En cada posición de la ventana (vuelta L), la variable vale lo que se sabía al inicio de la vuelta L, nunca después de la decisión. Medias y escalas se ajustan solo con train. Arquitectura y receta fijas: LSTM32 de una capa, dropout 0,2, BCE `pos_weight=2`, Adam 0,001, selección por F2. Se entrena la pareja con las semillas 42, 43 y 44 en la misma sesión CPU. El test 2025 permanece reservado.

## 1. Subir y verificar el paquete CPU

El paquete está en `runs/lstm_colab_estrategia_cpu_v1.zip` (regenerar con `python scripts/empaquetar_estrategia.py`; si lo regeneras, actualiza `EXPECTED_PACKAGE_SHA256`).

In [ ]:
from pathlib import Path
import hashlib, json, stat, zipfile
from google.colab import files

ROOT = Path('/content/f1-lstm-estrategia-cpu-v1')
PACKAGE = Path('/content/lstm_colab_estrategia_cpu_v1.zip')
EXPECTED_PACKAGE_SHA256 = 'eeba802d81798fa5bec99c49716f1bc61007d8815ff40d0f43653aa3a97622b2'
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-lstm-estrategia-cpu-v1 ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP CPU del experimento de estrategia.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != PACKAGE.name: raise ValueError(f'Se requiere {PACKAGE.name}.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP publicado.')
with zipfile.ZipFile(PACKAGE) as z:
    names = set()
    for m in z.infolist():
        p = Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest = json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_strategy_cpu_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    if names != set(manifest['files_sha256']) | {'colab_package_manifest.json'}: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT / name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; semillas', manifest['seeds'], '; sin datos de prueba.')

## 2. Instalar dependencias y confirmar runtime CPU

Ambas configuraciones fijan `device=cpu`. Son seis entrenamientos (tres parejas); en la pareja CPU anterior cada hijo tardó unos 4–5 minutos.

In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-estrategia-cpu-v1')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete extraído. Ejecuta primero la sección 1.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-colab-experiments.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available(), '| dispositivo efectivo: CPU')

## 3. Revisar las entradas

Solo lee los esquemas congelados y la cobertura de las variables nuevas en la última posición de la ventana. No entrena ni lee 2025.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-estrategia-cpu-v1')
sys.path.insert(0, str(ROOT / 'src'))
from f1_pitstop.rnn_data import read_json, verify_preprocessed
for role in ('ref', 'candidate'):
    cfg = read_json(ROOT / f'configs/experimento_lstm32_estrategia_{role}_cpu_seed42_v1.json')
    _, state = verify_preprocessed(ROOT / cfg['preprocessed_package'], cfg['preprocessed_manifest_sha256'])
    schema = state['schema']
    print(f"{role}: {cfg['preprocessed_package']} | numéricas: {len(schema['numeric_feature_names'])} | estáticas: {schema['static_feature_names']}")
report = read_json(ROOT / 'EDA/dataset/muestras_preprocesadas_v1_3_telemetria_estrategia_v1/report.json')
print('\nCobertura en la última posición (validación elegible):')
for name, share in report['splits']['validation']['present_share_last_slot_eligible'].items():
    print(f'  {name:30s} {share:6.1%}')

## 4. Entrenar las tres parejas en CPU

Cada pareja entrena primero la referencia (solo telemetría) y luego la candidata (telemetría + estrategia) con la misma semilla. El runner rechaza cualquier diferencia que no sea el paquete de entrada. Si una pareja falla, la celda imprime su `status.json` y se detiene.

In [ ]:
import json, subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-estrategia-cpu-v1')
SEEDS = (42, 43, 44)
RUNS = {seed: ROOT / f'runs/lstm32_estrategia_f2_cpu_v1_seed{seed}' for seed in SEEDS}
for seed, run in RUNS.items():
    if run.exists() or run.with_suffix('.zip').exists(): raise FileExistsError(f'La salida {run.name} ya existe. Usa una sesión nueva.')
for seed, run in RUNS.items():
    print(f'=== Semilla {seed} ===', flush=True)
    result = subprocess.run(
        [sys.executable, 'scripts/entrenar_comparacion_telemetria.py',
         '--reference-config', f'configs/experimento_lstm32_estrategia_ref_cpu_seed{seed}_v1.json',
         '--candidate-config', f'configs/experimento_lstm32_estrategia_candidate_cpu_seed{seed}_v1.json',
         '--output', str(run.relative_to(ROOT))],
        cwd=ROOT, check=False,
    )
    if result.returncode != 0:
        status_path = run / 'status.json'
        if status_path.is_file(): print(json.dumps(json.loads(status_path.read_text()), indent=2))
        raise RuntimeError(f'La pareja de la semilla {seed} falló con código {result.returncode}.')
print('Tres parejas completadas.')

## 5. Comparar resultados de validación 2024

Diferencia candidata − referencia por semilla y promedio. Como referencia externa, el LightGBM con telemetría + estrategia obtuvo AP 0,333 y F2 0,507 en la misma validación.

In [ ]:
import json
import pandas as pd
from pathlib import Path
ROOT = Path('/content/f1-lstm-estrategia-cpu-v1')
rows = []
for seed in (42, 43, 44):
    summary = json.loads((ROOT / f'runs/lstm32_estrategia_f2_cpu_v1_seed{seed}/pair_summary.json').read_text())
    for role, label in (('reference', 'solo telemetría'), ('candidate', 'telemetría + estrategia')):
        child = summary[role]
        selected = child['validation']['selected_threshold']
        rows.append({'semilla': seed, 'entradas': label, 'época': child['best_epoch'], 'umbral': child['threshold'],
                     'F2': selected['f2'], 'F1': selected['f1'], 'AP': selected['average_precision'],
                     'precisión': selected['precision'], 'recall': selected['recall'],
                     'FP': selected['fp'], 'FN': selected['fn']})
table = pd.DataFrame(rows)
display(table.round(4))
metrics = ['F2', 'F1', 'AP', 'precisión', 'recall', 'FP', 'FN']
wide = table.pivot(index='semilla', columns='entradas', values=metrics)
delta = pd.DataFrame({m: wide[(m, 'telemetría + estrategia')] - wide[(m, 'solo telemetría')] for m in metrics})
delta.loc['media'] = delta.mean()
print('Δ telemetría + estrategia − solo telemetría:')
display(delta.round(4))
print('Semillas con mejora de F2:', int((delta.loc[[42, 43, 44], 'F2'] > 0).sum()), 'de 3')

## 6. Descargar y devolver al proyecto

Descarga los tres ZIP `lstm32_estrategia_f2_cpu_v1_seed{42,43,44}.zip` y guárdalos intactos en `runs/`. Para la evaluación local de la candidata usa `--data EDA/dataset/muestras_preprocesadas_v1_3_telemetria_estrategia_v1` en `scripts/evaluar_rnn.py`. El test 2025 no se utilizó.

In [ ]:
from google.colab import files
from pathlib import Path
for seed in (42, 43, 44):
    files.download(str(Path('/content/f1-lstm-estrategia-cpu-v1') / f'runs/lstm32_estrategia_f2_cpu_v1_seed{seed}.zip'))